In [34]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

In [35]:
sqlite_index.search('How do I join the course?')

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives)."},
 {'id': '69d122f12e'

In [36]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [59]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches: First perform the search, then analyse the results and then perform more searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

assistant = RAGBase(sqlite_index, openai_client, instructions=instructions)

In [60]:
assistant.rag('How do I run Olama locally?')

'To run **Ollama locally**, follow the install steps for your operating system:\n\n### 1) Install Ollama\n- **macOS**: Download and install the `.pkg` from:  \n  https://ollama.com/download\n- **Windows**: Download and install the `.msi` from:  \n  https://ollama.com/download\n- **Linux**: Run:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n### 2) Start a model locally\nOnce installed, open a terminal and run:\n```bash\nollama run llama3\n```\n\nThis will:\n- download the model,\n- start it locally,\n- and open a chat-style prompt.\n\n### 3) Check that the server is running\nIn another terminal, test the local Ollama server:\n```bash\ncurl http://localhost:11434\n```\n\nYou should get a response like:\n```json\n{"models":[...]}\n```\n\n### 4) Use it from Python\nInstall the Python client:\n```bash\npip install ollama\n```\n\nMinimal example:\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": your

In [39]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [40]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [52]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered course join after start enrollment late join"}', call_id='call_DpUsDhLS8S5UyrqPPZFfMC4a', name='search', type='function_call', id='fc_0e59616da1bc5228006a36bfb81488819eabef3f0b233df8bf', namespace=None, status='completed')]

### At this point, the LLM has responded telling us to call our 'search' function with a specific query (the argument), so that is what we do here – and append the arg & result to the messages history

In [42]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [43]:
messages.append(call)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [44]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered course join after start enrollment eligibility"}', call_id='call_Wqn0CvFBtcLGj5X4JohCIUWL', name='search', type='function_call', id='fc_06ac4631047eaf36006a36b8c353a081919688193f04433789', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Wqn0CvFBtcLGj5X4JohCIUWL',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly wor

### We now append the search result to the messages history and again send this to the LLM

In [45]:
response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = messages,
    tools = [search_tool],
)

response.output_text

'Yes, you can still join.\n\nYou can start learning anytime, and the videos/materials are available. If you want a certificate, make sure you submit your project while submissions are still open.'

### Combining all this into agent loops:

In [55]:
# Helper for calling tools (function calls)
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [49]:
messages.extend(response.output)

# check if an llm response is a tool/function call or a message. Function calls will have to be actioned.
for item in response.output:
    if item.type == 'function_call':
        print('function call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT: ')
        print(item.content[0].text)

ASSISTANT: 
Yes, you can still join.

You can start learning anytime, and the videos/materials are available. If you want a certificate, make sure you submit your project while submissions are still open.


In [50]:
# Combining this all into one Agentic Loop

In [ ]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    # reset messages:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    i = 1

    while True:
        print(f'iteration #{i}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT: ')
                last_answer = item.content[0].text
                print(item.content[0].text)

        i += 1
        if has_function_calls == False:
            break

    return last_answer

iteration #1...
function call: search {"query":"join course discovered late can I join enrollment registration access late join"}
iteration #2...
function call: search {"query":"certificate self-paced live cohort project peer-review submissions open form closed join course late start learning homework while form is open"}
iteration #3...
ASSISTANT: 
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted. If you’re only learning and not aiming for the certificate, you can start anyway.

If you want, I can also help you figure out how to get started or what the current course rules are.


### Applying all this within a framework (using ToyAIKit as a simplified example)

In [65]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
# define tools
def search(query: str):
    """
    Search the FAQ database for entries matching the given query.
    """
    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section':0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [ ]:
# add tools
agent_tools = Tools()
agent_tools.add_tool(search)

In [68]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [71]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [72]:
# define agent
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [73]:
result = runner.loop(
    prompt = 'How do I run Ollama locally?',
    callback = callback
)

-> Response received


-> Response received


-> Response received


In [74]:
result.cost

CostInfo(input_cost=Decimal('0.00276075'), output_cost=Decimal('0.001476'), total_cost=Decimal('0.00423675'))

In [75]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches: First perform the search, then analyse the results and then perform more searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Ollama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"run Ollama locally install start Ollama local machine"}', call_id='call_bN55X8Tir7GIjfBfJonft0Qv', name='search', type='function_call', id='fc_0f515d28c93167a8006a36c8a495a8819ea443e25dd7840de0', namespace=None, status='completed'),
 {'type

In [76]:
# for continuing the conversation:
result2 = runner.loop(
    prompt = 'How do I run a different model?',
    previous_messages = result.all_messages,
    callback = callback
)

-> Response received


-> Response received


In [77]:
runner.run();

-> Response received


-> Response received


-> Response received


Chat ended.
